# Task 1 — Tokenize and compare

In [ ]:
from transformers import AutoTokenizer
import tiktoken
import pandas as pd
TEXT = "Hello! `print('world')` 🌍. ¡Hola, cómo estás? 12345."

# Tiktoken (GPT-4o style)
enc1 = tiktoken.get_encoding("cl100k_base")
ids1 = enc1.encode(TEXT)
tokens1 = [enc1.decode([i]) for i in ids1]

# HuggingFace (Llama 3 style)
enc2 = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B")
ids2 = enc2.encode(TEXT)
tokens2 = enc2.convert_ids_to_tokens(ids2)

import pandas as pd
df = pd.DataFrame({"Tokenizer": ["tiktoken (cl100k)", "Llama 3 (HF)"], "Count": [len(ids1), len(ids2)]})
print(df)
print("\nTokens (Tiktoken):", tokens1)
print("Tokens (Llama 3):", tokens2)

Where do the two tokenizers disagree most, and why?

The tokenizers diverge most on non-English characters (e.g., "¡" and "á") and emojis. This occurs because tokenizers are trained on different base corpora; cl100k_base is optimized for modern web-text including broad Unicode support, while Llama 3's vocabulary may treat specific character combinations as individual tokens versus multiple sub-words depending on their frequency in their respective training datasets.

# Task 2 — Hunt the weird cases

In [ ]:
cases = {
    "common_word": "running",
    "rare_word": "antidisestablishmentarianism",
    "english": "The cat sat on the mat.",
    "hindi": "बिल्ली चटाई पर बैठी थी।",
    "code": "def func(): return True",
    "prose": "define a function that returns true",
    "space_no": "hello",
    "space_yes": " hello"
}

results = []
for name, s in cases.items():
    results.append({"Case": name, "Text": s, "Count": len(enc1.encode(s))})

print(pd.DataFrame(results))

Observations:

Rare vs. Common: "Running" is a single token, while the rare word is split into five, showing how vocabulary limits increase costs for specialized technical or obscure terminology.

English vs. Hindi: The Hindi sentence consumes significantly more tokens than the English equivalent because the tokenizer vocabulary is heavily biased toward Latin characters, forcing the model to decompose Devanagari script into many small sub-units.

Code vs. Prose: Code often results in a higher token count than semantically equivalent prose due to the frequent use of special characters (parentheses, brackets) which often act as individual tokens.

Leading Space: A leading space often acts as its own token, which illustrates why inconsistent formatting or extra whitespace can slightly inflate context window usage.

Task 3 — Token + cost estimator

In [ ]:
def estimate(text, price_in_per_1k, price_out_per_1k, expected_output_tokens):
    n_in = len(enc1.encode(text))
    cost = (n_in / 1000) * price_in_per_1k + (expected_output_tokens / 1000) * price_out_per_1k
    return n_in, cost

inputs = {
    "short_question": "What is the capital of Azerbaijan?",
    "long_document": "The history of Baku spans centuries..." * 50,
    "code_file": "def calculate_area(r):\n    return 3.14 * r * r\n" * 20
}

# Pricing: Input $0.05/1k, Output $0.15/1k
for name, text in inputs.items():
    n_in, cost = estimate(text, 0.05, 0.15, 100)
    print(f"{name:15} | Tokens: {n_in:4} | Cost: ${cost:.4f}")

Which input is most expensive, and is it what you expected?

The "long_document" is the most expensive; this is expected as cost scales linearly with input length, demonstrating why summarizing long-form content is significantly more expensive than processing single queries.